# py-condiments — Tutorial on synthetic data

Walkthrough of all 7 public functions.

## 1. What this package does

`pycondiments` is a port of [condiments](https://github.com/HectorRDB/condiments) (Roux de Bezieux et al. *Nat Commun* 2024) — differential-trajectory analysis between conditions in scRNA-seq.

Given a Slingshot trajectory + condition labels, it tests:
- `imbalance_score` — per-cell condition-imbalance (kNN multinomial test)
- `topologyTest` — does topology differ between conditions?
- `progressionTest` — different pseudotime rates per condition?
- `differentiationTest` — different lineage choices per condition?

**Citation**: Roux de Bezieux, H. et al. *Nature Communications* 15, 1281 (2024)

## 2. Install + import

In [1]:
import sys
sys.path.insert(0, '/scratch/users/steorra/analysis/omicverse_traj_dev/py-condiments')
import pycondiments as cd
print(f'pycondiments {cd.__version__}')

pycondiments 0.1.0


## 3. Synthetic data

In [2]:
import numpy as np
import matplotlib.pyplot as plt
toy = cd.create_differential_topology(n_cells=400, shift=0.7, seed=42)
print(f"pseudotime: {toy['pseudotime'].shape}  cellWeights: {toy['cellWeights'].shape}")
print(f"conditions: {np.unique(toy['conditions'], return_counts=True)}")

pseudotime: (400, 2)  cellWeights: (400, 2)
conditions: (array(['0', '1'], dtype='<U21'), array([207, 193]))


## 4. Public functions

### 4.1 imbalance_score — per-cell condition imbalance

In [3]:
# We need a 2D reduced-space; use the first cellWeights column + jitter as a fake "rd"
rng = np.random.default_rng(42)
rd = np.column_stack([toy['cellWeights'][:,0] + rng.normal(0,0.1,400),
                      toy['cellWeights'][:,1] + rng.normal(0,0.1,400)])
imb = cd.imbalance_score(rd, toy['conditions'], k=10, smooth=10)
print(f"score:        min={imb['score'].min():.2f}  max={imb['score'].max():.2f}  mean={imb['score'].mean():.2f}")
print(f"scaled_score: min={imb['scaled_score'].min():.2f}  max={imb['scaled_score'].max():.2f}")

score:        min=0.00  max=3.59  mean=2.17
scaled_score: min=0.68  max=3.45


### 4.2 progressionTest

In [4]:
ptest = cd.progressionTest(toy['pseudotime'], toy['conditions'], cellWeights=toy['cellWeights'])
print(ptest.to_string())

   statistic    pvalue
0   4.047253  0.399649


### 4.3 differentiationTest

In [5]:
dtest = cd.differentiationTest(toy['cellWeights'], toy['conditions'])
print(dtest.to_string())

    statistic        pvalue
0  383.296149  1.129961e-81


### 4.4 topologyTest

In [6]:
ttest = cd.topologyTest(toy['pseudotime'], toy['cellWeights'], toy['conditions'])
print(f"p-value: {ttest['pvalue']:.4g}")
print(f"chi² statistic: {ttest['statistic']:.4f}")

p-value: 9.935e-40
chi² statistic: 173.9928


### 4.5 weights_from_pst + merge_sds

In [7]:
cw_derived = cd.weights_from_pst(toy['pseudotime'])
print(f"derived weights shape: {cw_derived.shape}; matches input? {np.allclose(cw_derived, toy['cellWeights'])}")

# merge_sds:
a = {'pseudotime': toy['pseudotime'][:200], 'cellWeights': toy['cellWeights'][:200]}
b = {'pseudotime': toy['pseudotime'][200:], 'cellWeights': toy['cellWeights'][200:]}
merged = cd.merge_sds(a, b)
print(f"merged: pt {merged['pseudotime'].shape}, cw {merged['cellWeights'].shape}")

derived weights shape: (400, 2); matches input? True
merged: pt (400, 2), cw (400, 2)


## 5. Common pitfalls / FAQ

- **`topologyTest` v0.1 is approximate**: uses χ² on dominant-lineage contingency, not Slingshot re-fitting. Strong topology shifts (e.g., extra lineage in one condition) will be missed.
- **`differentiationTest` needs ≥ 2 lineages**: errors on single-lineage trajectories.
- **`imbalance_score`'s `smooth` parameter**: R uses GAM; Py uses kNN-average. Both give similar smoothed z-scores.

## 6. Next
- [`README.md`](../README.md), [`compare_R_vs_Python.ipynb`](compare_R_vs_Python.ipynb), [`function_by_function_R_parity.ipynb`](function_by_function_R_parity.ipynb)